# NB8: API Boundaries & Asynchronous Orchestration

**2-minute intro script:** A production multi-agent team does not live inside `if __name__ == "__main__"`. It receives GitHub webhooks, Slack commands, Jira tickets, and A2A HTTP requests. The API boundary is the governance perimeter: messy external JSON arrives, identity is checked, policy is enforced, and only then does the request become a strict internal task. Because agent workflows may take seconds or minutes, the endpoint returns `202 Accepted` immediately and the pipeline runs in the background.

## Management Principle

The API gateway is the customs border for your AI workforce. External systems can ask for work, but they do not get direct access to internal agents, tools, memory, or release actions. Every request must cross identity, policy, and schema checks first.

In [ ]:
import asyncio
from typing import Any, Literal
from uuid import uuid4

from fastapi import BackgroundTasks, Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel, ConfigDict, Field

from src.enterprise_agent_team import product_manager_parse_issue, run_virtual_software_company

app = FastAPI(title="Virtual Software Company API")

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

# External world: intentionally small, messy, and untrusted.
class ExternalIssueRequest(StrictModel):
    github_issue_url: str
    priority: str
    requester_email: str

# Internal identity: external API keys map into zero-trust identities.
class AgentIdentity(StrictModel):
    role: Literal["product_manager", "vendor_reporter"]
    trust_tier: Literal["public", "confidential"]

class TaskStatusResponse(StrictModel):
    task_id: str
    status: Literal["PENDING", "RUNNING", "SHIPPED", "ESCALATED"]
    pull_request_summary: dict[str, Any] | None = None

# Teaching adapter: production uses Redis, Postgres, or a durable queue.
task_store: dict[str, dict[str, Any]] = {}

def get_requester_identity(authorization: str = Header(default="")) -> AgentIdentity:
    """Map external API keys to internal zero-trust identities."""
    token = authorization.removeprefix("Bearer ").strip()
    if token == "sk-internal-admin":
        return AgentIdentity(role="product_manager", trust_tier="confidential")
    if token == "sk-external-vendor":
        return AgentIdentity(role="vendor_reporter", trust_tier="public")
    raise HTTPException(status_code=401, detail="Invalid API key")

In [ ]:
async def run_company_async(task_id: str, issue_url: str, identity: AgentIdentity) -> None:
    """Long-running multi-agent pipeline triggered by the API boundary."""
    task_store[task_id]["status"] = "RUNNING"

    # PM converts external request context into a strict internal TaskSpec.
    task_spec = product_manager_parse_issue(
        f"Task requested by {identity.role} for {issue_url}"
    )

    # Simulate LLM/tool latency. Production uses queues/workers.
    await asyncio.sleep(0.01)

    final_state = run_virtual_software_company(task_spec.goal)
    review = final_state.review.model_dump(mode="json") if final_state.review else {}
    status = "SHIPPED" if review.get("next_action") == "ship" else "ESCALATED"

    task_store[task_id]["status"] = status
    task_store[task_id]["pull_request_summary"] = {
        "task": final_state.task.model_dump(mode="json") if final_state.task else None,
        "route_trace": [route.model_dump(mode="json") for route in final_state.route_trace],
        "review": review,
        "repair_attempts": final_state.repair_attempts,
    }

@app.post("/tasks", response_model=TaskStatusResponse, status_code=202)
async def ingest_issue(
    request: ExternalIssueRequest,
    background_tasks: BackgroundTasks,
    identity: AgentIdentity = Depends(get_requester_identity),
) -> TaskStatusResponse:
    # Governance rule: public vendors cannot create high-risk internal work.
    if identity.trust_tier == "public" and request.priority.lower() == "high":
        raise HTTPException(
            status_code=403,
            detail="Vendors cannot set high priority.",
        )

    task_id = f"task-{uuid4().hex[:8]}"
    task_store[task_id] = {
        "task_id": task_id,
        "status": "PENDING",
        "pull_request_summary": None,
    }

    background_tasks.add_task(
        run_company_async,
        task_id,
        request.github_issue_url,
        identity,
    )

    return TaskStatusResponse(task_id=task_id, status="PENDING")

def read_task_status(task_id: str) -> dict[str, Any]:
    if task_id not in task_store:
        raise HTTPException(status_code=404, detail="Task not found")
    return task_store[task_id]

@app.get("/tasks/{task_id}", response_model=TaskStatusResponse)
async def get_task(task_id: str) -> dict[str, Any]:
    return read_task_status(task_id)

@app.get("/status/{task_id}", response_model=TaskStatusResponse)
async def get_status(task_id: str) -> dict[str, Any]:
    # Teaching alias: many systems call this a status endpoint.
    return read_task_status(task_id)

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

print("--- Test 1: Internal admin triggers high-priority task ---")
response = client.post(
    "/tasks",
    json={
        "github_issue_url": "https://github.com/org/repo/issues/42",
        "priority": "high",
        "requester_email": "pm@company.com",
    },
    headers={"Authorization": "Bearer sk-internal-admin"},
)
print(f"Status: {response.status_code} (should be 202)")
task_id = response.json()["task_id"]
assert response.status_code == 202

print("\n--- Test 2: Poll status endpoint ---")
status_response = client.get(f"/tasks/{task_id}")
print(status_response.json())
assert status_response.status_code == 200
assert status_response.json()["status"] in {"PENDING", "RUNNING", "SHIPPED", "ESCALATED"}

print("\n--- Test 3: External vendor blocked from high priority ---")
blocked_response = client.post(
    "/tasks",
    json={
        "github_issue_url": "https://github.com/external/repo/issues/7",
        "priority": "high",
        "requester_email": "vendor@external.com",
    },
    headers={"Authorization": "Bearer sk-external-vendor"},
)
print(f"Status: {blocked_response.status_code} (should be 403)")
print(f"Reason: {blocked_response.json()['detail']}")
assert blocked_response.status_code == 403

print("\n--- Test 4: Invalid API key blocked ---")
bad_key_response = client.post(
    "/tasks",
    json={
        "github_issue_url": "https://github.com/org/repo/issues/1",
        "priority": "low",
        "requester_email": "unknown@example.com",
    },
    headers={"Authorization": "Bearer bad-key"},
)
print(f"Status: {bad_key_response.status_code} (should be 401)")
assert bad_key_response.status_code == 401

## 🧪 Exercises: The Front Door

**The Story:** Your multi-agent system is working perfectly in your Jupyter Notebook. But now, the VP of Engineering wants to trigger it from a Slack command, and the Product team wants to trigger it from a GitHub webhook. And they want it to scale to 1,000 concurrent requests.

**Your Mission:**
1. **The GitHub Bridge:** Add a `/webhooks/github` alias endpoint. It must parse the raw GitHub JSON payload and map it into the same internal `TaskStatusResponse`.
2. **The Legacy Client:** Add an `X-API-Key` header variant as an alternative to `Authorization: Bearer`. Many internal tools still use legacy auth headers.
3. **The Audit Trail:** Store `created_by`, `priority`, and `requester_email` in the `task_store`. When the task completes, include these in the final `PullRequestSummary`.
4. **The Perimeter Check:** Add a denied-memory-access test. Prove that a `vendor_reporter` identity cannot query the shared memory for internal tasks.
5. **The Production Reality:** Replace the in-memory `task_store` dictionary with a design note for Redis or PostgreSQL. How would you handle concurrent writes to the same `task_id`?

**The Takeaway:** The API gateway is the customs border for your AI workforce. External systems can ask for work, but they do not get direct access to your agents.